# 08 - SQL, Excel & Power BI

This last notebook prepares the outputs for simple reporting and dashboard work.

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
daily = pd.read_csv("../01_Data/processed/daily_energy.csv")
hourly = pd.read_csv("../01_Data/processed/hourly_energy.csv")

## Put the data into SQLite

In [ ]:
con = sqlite3.connect("../03_Analytics_BI_SQL/sql/household_energy.db")

In [ ]:
daily.to_sql("daily_energy", con, if_exists="replace", index=False)
hourly.to_sql("hourly_energy", con, if_exists="replace", index=False)

## One simple SQL business question

In [ ]:
query = """
SELECT dayofweek, AVG(daily_kwh) AS avg_kwh
FROM daily_energy
GROUP BY dayofweek
ORDER BY dayofweek
"""

In [ ]:
sql_result = pd.read_sql(query, con)
sql_result

## Another SQL check

In [ ]:
query = """
SELECT month, AVG(daily_kwh) AS avg_kwh
FROM daily_energy
GROUP BY month
ORDER BY month
"""

In [ ]:
monthly = pd.read_sql(query, con)
monthly.head()

## Excel handoff

In [ ]:
weekday = daily.loc[daily.is_weekend == 0, "daily_kwh"].mean()
weekend = daily.loc[daily.is_weekend == 1, "daily_kwh"].mean()
uplift = (weekend / weekday - 1) * 100

kpi = pd.DataFrame({
    "Metric": ["Avg daily kWh", "Weekend uplift", "Usable days"],
    "Value": [daily["daily_kwh"].mean(), uplift, len(daily)]
})


In [ ]:
kpi.to_excel(
    "../03_Analytics_BI_SQL/excel/energy_kpi_summary.xlsx",
    index=False
)

## Power BI data

I keep the dashboard inputs simple: daily energy for KPIs and trends, hourly energy for the time profile.

In [ ]:
daily[["date", "daily_kwh", "is_weekend", "month", "dayofweek"]].to_csv(
    "../03_Analytics_BI_SQL/powerbi/powerbi_daily.csv", index=False
)

In [ ]:
hourly[["date_time", "energy_kwh", "hour", "is_weekend"]].to_csv(
    "../03_Analytics_BI_SQL/powerbi/powerbi_hourly.csv", index=False
)

In [ ]:
con.close()

### Dashboard idea

KPI cards → Daily trend → Hourly usage → Weekday vs weekend → Monthly trend → High-use hours.

The project finishes with business and sustainability actions based on these outputs.